# 25A. Экспорт пула C2 top-200 для GPU-подбора Recall@50 objective

Notebook 24 зафиксировал пул attempt 6: `BM25 + zero-shot BGE-M3 + LoRA v2`,
union top-100 каналов, обрезанный до top-200 по RRF-признаку `fused`.
Selector в нём обучался `YetiRankPairwise`, который не знает про cutoff 50.

Stage 25B на Kaggle (2×T4) перебирает objective и параметры CatBoost. Этот
notebook только готовит для него неизменённый вход, поэтому признаки, split и
baseline берутся строго из notebook 19 (запускается как prerequisite, как в
notebook 24). Labels экспортируются полностью, включая positives вне пула:
знаменатель macro Recall@50 должен совпадать с offline-оценкой репозитория.

Выходы (`artifacts/recall50_kaggle_input/`, не коммитятся, загружаются как
private Kaggle dataset):

- `c2_validation.parquet`, `c2_benchmark.parquet` — признаки пула;
- `relevant.parquet` — все relevant пары holdout;
- `splits.json` — dev/test/tail и 2-fold cross-fitting на dev;
- `attempt4_test_top50.parquet` — test-ранжирование attempt 4 (C3) для
  локального статистического gate в 25C; на Kaggle не используется.

In [1]:
from __future__ import annotations

from pathlib import Path
import hashlib
import json

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

# Воспроизводим признаки и модели attempt 4 тем же кодом, что notebook 24.
get_ipython().run_line_magic("run", str(ROOT / "notebooks/19_lora_v2_evaluation.py"))

TOP_POOL = 200  # как в notebook 24: буквальное сжатие 200 → 50
OUT = ROOT / "artifacts/recall50_kaggle_input"
OUT.mkdir(parents=True, exist_ok=True)
C2_FEATURES = feature_columns("C2")

{'stage': 'retrieval_v2',
 'base_model': 'deepvk/USER-bge-m3',
 'license': 'apache-2.0',
 'training': {'seed': 42,
  'batch_per_gpu': 32,
  'lora_rank': 32,
  'learning_rate': 0.0002,
  'temperature': 0.03,
  'checkpoint_every': 1000,
  'pairs': 457439,
  'steps': 7147,
  'mean_loss': 0.36985957852785106,
  'masked_false_negatives_rank0': 6680,
  'elapsed_seconds': 17496.136995561,
  'trainable_parameters': 14155776,
  'world_size': 2},
 'items': 189212,
 'passages': 554920,
 'queries': 4904,
 'chunk_words': 140,
 'overlap': 30,
 'max_chunks': 4,
 'embedding_dtype': 'float16',
 'gpus': ['Tesla T4', 'Tesla T4']}

v2 local weight selected on dev: 2.0 {1.0: 0.8048531810766721, 1.15: 0.8198069603045134, 1.5: 0.8317020119630234, 2.0: 0.8333333333333333}


,segment,k,v1,v2,mean_delta,ci_low,ci_high,p_value_greater,n_queries
0,test,50,0.7998,0.8268,0.0270,0.0116,0.0428,0.0004,1226
1,test,250,0.9193,0.9381,0.0188,0.0077,0.0302,0.0006,1226
2,test_tail,50,0.7831,0.8042,0.0212,-0.0265,0.0741,0.2673,189
3,test_tail,250,0.9365,0.9418,0.0053,-0.0212,0.0317,0.5001,189
4,test_head,50,0.8028,0.8309,0.0281,0.0124,0.0443,0.0002,1037
5,test_head,250,0.9162,0.9374,0.0212,0.0090,0.0338,0.0003,1037


C1 (495370, 36) pool positives: 2490


C2 (513806, 36) pool positives: 2502


...

,system,segment,selected,recall,c1_recall,mean_delta,ci_low,ci_high,p_value_greater,n_queries,wins,losses
0,C2,test_tail,False,0.90476,0.91005,-0.00529,-0.03704,0.02646,0.77461,189,3,4
1,C2,test,False,0.86052,0.84829,0.01223,-0.00027,0.02574,0.01230,1226,33,16
2,C3,test_tail,True,0.91005,0.91005,0.00000,-0.03175,0.03175,0.65797,189,3,3
3,C3,test,True,0.86120,0.84829,0.01291,0.00136,0.02501,0.00515,1226,29,11
4,C4,test_tail,False,0.88360,0.91005,-0.02646,-0.07407,0.01587,0.95410,189,4,9
5,C4,test,False,0.85651,0.84829,0.00822,-0.00659,0.02291,0.09435,1226,38,25


{'alpha': 0.016666666666666666, 'selected': 'C3', 'test_tail': False, 'test': True}


,segment,candidate,actual_attempt_3,mean_delta,ci_low,ci_high,p_value_greater,n_queries
0,test_tail,0.91005,0.90476,0.00529,-0.02646,0.03704,0.50227,189
1,test,0.86120,0.84448,0.01672,0.00489,0.02936,0.00080,1226


{'deployment_safe': True}


benchmark overlap with candidate 3: 0.840
C4 overlap with selected C3: 0.864
saved reports/lora_v2_metrics.json


## 1. Validation и benchmark пулы

Ограничение `fused <= 200` не использует labels и одинаково применяется к
holdout и benchmark. Порядок строк внутри запроса — по `fused`, чтобы
tie-break на Kaggle совпадал с notebook 24.

In [2]:
def top_pool(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame[frame.fused.le(TOP_POOL)]
    return frame.sort_values(["query_key", "fused"], kind="stable").reset_index(drop=True)


c2_validation = top_pool(features_by_system["C2"])
c2_benchmark = top_pool(pd.concat([
    system_features("C2", ("benchmark", str(query_id)))
    for query_id in benchmark_queries.query_id
], ignore_index=True))
c2_validation[["query_key", "item_id", "label", *C2_FEATURES]].to_parquet(
    OUT / "c2_validation.parquet", index=False)
c2_benchmark[["query_key", "item_id", *C2_FEATURES]].to_parquet(
    OUT / "c2_benchmark.parquet", index=False)
print("validation", c2_validation.shape, "benchmark", c2_benchmark.shape)

validation (427213, 36) benchmark (423847, 35)


## 2. Labels, split и test-ранжирование attempt 4

In [3]:
labels[["eval_query_id", "item_id"]].astype(str).to_parquet(OUT / "relevant.parquet", index=False)
splits = {
    "features": C2_FEATURES,
    "dev": sorted(dev_ids), "test": sorted(test_ids), "tail": sorted(tail_ids),
    "cross_folds": [{"fit": sorted(fit), "held": sorted(held)} for fit, held in cross_folds],
    "benchmark_query_ids": [str(q) for q in benchmark_queries.query_id],
}
(OUT / "splits.json").write_text(json.dumps(splits), encoding="utf-8")

attempt4 = pd.DataFrame([
    {"query_id": query_id, "rank": rank, "item_id": item_id}
    for query_id, items_ in test_predictions["C3"].items()
    for rank, item_id in enumerate(items_, 1)
])
attempt4.to_parquet(OUT / "attempt4_test_top50.parquet", index=False)
(OUT / "dataset-metadata.json").write_text(json.dumps({
    "title": "Avito Recall50 Selector Pool",
    "id": "m1r0tvorxc/avito-recall50-selector-pool",
    "licenses": [{"name": "other"}],
}, indent=2), encoding="utf-8")

for path in sorted(OUT.glob("*.parquet")) + [OUT / "splits.json"]:
    print(path.name, hashlib.sha256(path.read_bytes()).hexdigest()[:16])

attempt4_test_top50.parquet 3ce199d7f2943d44
c2_benchmark.parquet 9fde5b8bf10b92d5
c2_validation.parquet 55eb498421650d62
relevant.parquet 8eaf3015aa4602aa
splits.json 38b18450eb7d4626
